In [4]:
def smart_home_agent(temp, is_dark, person_in_room):
    actions = []

    if temp > 30:
        actions.append("AC is ON")
    elif temp < 18:
        actions.append("Heater is ON")
    else:
        actions.append("Temperature is Ok")

    if is_dark and person_in_room:
        actions.append("light turned ON")
    elif not person_in_room:
        actions.append("Light is OFF")

    return actions

In [5]:
percept = [
    (35, True,  True),
    (15, True,  True),
    (24, False, True),
    (28, True,  False),
    (32, False, False),
]

for p in percept:
    result = smart_home_agent(p[0], p[1], p[2])
    print("percept : ", p, "->", result)

percept :  (35, True, True) -> ['AC is ON', 'light turned ON']
percept :  (15, True, True) -> ['Heater is ON', 'light turned ON']
percept :  (24, False, True) -> ['Temperature is Ok']
percept :  (28, True, False) -> ['Temperature is Ok', 'Light is OFF']
percept :  (32, False, False) -> ['AC is ON', 'Light is OFF']


In [8]:
expected_first_action = ["AC is ON", "Heater is ON", "Temprature is OK", "AC is ON", "Heater is ON"]

score = 0
for i in range(len(percept)):
    p = percept[i]
    action = smart_home_agent(p[0], p[1], p[2])[0]
    if action == expected_first_action[i]:
        score = score + 1
print("Score", score, "Out of", len(percept))
    

Score 2 Out of 5


In [9]:
state = {"ac_on" : False, "heater_on" : False, "light_on" : False, "last_temp" : 25}
def model_based_agent(temp, is_dark, person_in_room):
    actions = []
    if temp is None:
        temp = state["last_temp"]
    state["last_temp"] = temp

    if temp > 30 and not state["ac_on"]:
        actions.append("AC is ON")
        state["ac_on"] = True
        state["heater_on"] = False
    elif temp < 18 and not state["heater_on"]:
        actions.append("heater is ON")
        state["ac_on"] = False
        state["heater_on"] = True
    elif 18 <= temp <= 30 and (state["ac_on"] or state["heater_on"]):
        actions.append("AC/heater turned OFF")
        state["ac_on"] = False
        state["heater_on"] = False

    if is_dark and person_in_room and not state["light_on"]:
        actions.append("Light is ON")
        state["light_on"] = True
    elif not person_in_room and state["light_on"]:
        actions.append("Light is OFF")
        state["light_on"] = False

    if len(actions) == 0:
        actions.append("NO changes happened")
    return actions

In [10]:
sequence = [
    (35, True, True),
    (36, True, True),
    (None, True, True),
    (24, False, False),
    (14, True, True)
]

for p in sequence:
    if p[0] is None:
        reflex = "Cannot decide without memory "
    else:
        reflex = smart_home_agent(p[0], p[1], p[2])
    model = model_based_agent(p[0], p[1], p[2])

    print("percept             :", p)
    print("simple reflex agent :", reflex)
    print("Model based agent   :", model)

percept             : (35, True, True)
simple reflex agent : ['AC is ON', 'light turned ON']
Model based agent   : ['AC is ON', 'Light is ON']
percept             : (36, True, True)
simple reflex agent : ['AC is ON', 'light turned ON']
Model based agent   : ['NO changes happened']
percept             : (None, True, True)
simple reflex agent : Cannot decide without memory 
Model based agent   : ['NO changes happened']
percept             : (24, False, False)
simple reflex agent : ['Temperature is Ok', 'Light is OFF']
Model based agent   : ['AC/heater turned OFF', 'Light is OFF']
percept             : (14, True, True)
simple reflex agent : ['Heater is ON', 'light turned ON']
Model based agent   : ['heater is ON', 'Light is ON']
